## Step 1: S3 Data Storage Setup
- List available S3 buckets.
- Choose or create buckets for raw input and processed output.
- Configure S3 event notifications (if needed).

In [1]:
# List available S3 buckets using boto3
import boto3
s3 = boto3.client('s3')
response = s3.list_buckets()
for bucket in response['Buckets']:
    print(bucket['Name'])

elbee-oreumi


## Step 2: Upload Sample Raw Data to S3
- Generate or use sample JSON data.
- Upload to the raw data S3 bucket.

In [2]:
# Example: Generate and upload sample JSON data to S3
import json
import random
import datetime
sample_data = [{
    'transaction_id': i,
    'customer_id': random.randint(1000, 9999),
    'product_category': random.choice(['A', 'B', 'C']),
    'amount': round(random.uniform(10, 500), 2),
    'timestamp': (datetime.datetime.now() - datetime.timedelta(days=random.randint(0, 30))).isoformat(),
    'region': random.choice(['US', 'EU', 'ASIA'])
} for i in range(10)]
with open('sample_data.json', 'w') as f:
    json.dump(sample_data, f, indent=2)
# Upload to S3 (replace 'elbee-oreumi' and 'raw/sample_data.json' as needed)
s3.upload_file('sample_data.json', 'elbee-oreumi', 'raw/sample_data.json')

## Step 3: Lambda Data Processor Setup
- Outline Lambda function requirements.
- (Implementation may be done in AWS Console or as a script/template.)

In [4]:
# Lambda function for S3-triggered processing (Free Tier compatible)
import json
import boto3
import os
from datetime import datetime

s3 = boto3.client('s3')
dynamodb = boto3.resource('dynamodb')
DDB_TABLE = os.environ.get('DDB_TABLE')
PROCESSED_BUCKET = os.environ.get('PROCESSED_BUCKET')

def validate_record(data):
    required = ['transaction_id', 'customer_id', 'amount']
    for field in required:
        if field not in data:
            return False
    return True

def enrich_record(data):
    data['processed_at'] = datetime.utcnow().isoformat()
    data['source'] = 'lambda-s3-processor'
    return data

def route_record(data):
    if DDB_TABLE:
        table = dynamodb.Table(DDB_TABLE)
        table.put_item(Item=data)
    if PROCESSED_BUCKET:
        key = f"processed/{data['transaction_id']}.json"
        s3.put_object(Bucket=PROCESSED_BUCKET, Key=key, Body=json.dumps(data))

def lambda_handler(event, context):
    for record in event['Records']:
        bucket = record['s3']['bucket']['name']
        key = record['s3']['object']['key']
        obj = s3.get_object(Bucket=bucket, Key=key)
        raw_data = json.loads(obj['Body'].read())
        # If the file contains a list of records
        if isinstance(raw_data, list):
            records = raw_data
        else:
            records = [raw_data]
        for data in records:
            if not validate_record(data):
                print("Validation failed, skipping:", data)
                continue
            enriched = enrich_record(data)
            route_record(enriched)
    return {'statusCode': 200, 'body': 'Processing complete'}

In [5]:
# --- Test the Lambda handler locally with a mock event ---
import os
import json
from unittest.mock import MagicMock

# Set environment variables for testing (replace with your test bucket/table or leave as None)
os.environ['DDB_TABLE'] = ''  # or your DynamoDB table name
os.environ['PROCESSED_BUCKET'] = ''  # or your processed S3 bucket name

# Create a mock S3 client and resource if you want to avoid real AWS calls
# For now, this will use your real AWS credentials/config

# Prepare a sample S3 event (simulate S3 put event)
sample_event = {
    "Records": [
        {
            "s3": {
                "bucket": {"name": "elbee-oreumi"},
                "object": {"key": "raw/sample_data.json"}
            }
        }
    ]
}

# Optionally, patch boto3 to avoid real AWS calls (advanced, not shown here)

# Run the handler and print the result
try:
    result = lambda_handler(sample_event, MagicMock())
    print("Lambda handler result:", result)
except Exception as e:
    print("Error during Lambda handler execution:", e)


Lambda handler result: {'statusCode': 200, 'body': 'Processing complete'}


## Step 4: Data Format Conversion and Monitoring
- Convert JSON to CSV.
- Set up CloudWatch monitoring.